# H2O AutoML: Enterprise-Grade Automated Machine Learning

## What Is H2O?

Imagine a factory that makes cars. Instead of one worker doing everything, you have a production line —  
each station specializes in one part (engine, wheels, paint), and a supervisor assembles the best final car.

**H2O AutoML** is that automated factory for ML models:  
It trains many models in parallel, stacks the best ones, and delivers a ranked leaderboard.

**H2O.ai** is an open-source ML platform:
- **H2O** (open source): distributed in-memory ML, AutoML
- **H2O-3**: the core engine — runs on Java, accessible from Python/R
- **H2O Driverless AI**: commercial AutoML product
- **H2O Wave**: ML app framework

H2O is used at AT&T, Comcast, PayPal, and thousands of enterprises for large-scale ML.

Key capabilities:
- **AutoML**: trains GBM, XGBoost, RF, GLM, Deep Learning, stacked ensembles automatically
- **Distributed**: runs on a cluster of machines (not just your laptop)
- **MOJO export**: deploy models as fast Java objects — no H2O dependency at serving time
- **Explainability**: built-in SHAP, partial dependence plots, model explanations

## Resources

- **Docs**: [https://docs.h2o.ai/](https://docs.h2o.ai/)
- **GitHub**: [https://github.com/h2oai/h2o-3](https://github.com/h2oai/h2o-3)
- **YouTube**: [https://www.youtube.com/watch?v=LM255qs8Zsk](https://www.youtube.com/watch?v=LM255qs8Zsk)
- **Free courses**: [https://h2o.ai/resources/](https://h2o.ai/resources/)

## Installation

```bash
pip install h2o
# H2O requires Java 8+ (not 11 or 17 — check your version!)
# java -version

# If Java not installed:
#   macOS:   brew install openjdk@11
#   Ubuntu:  sudo apt install default-jdk
#   Windows: https://adoptium.net/
```

In [ ]:
import numpy as np
import time

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

try:
    import h2o
    from h2o.automl import H2OAutoML
    H2O_AVAILABLE = True
    print(f"H2O version: {h2o.__version__}")
except ImportError:
    H2O_AVAILABLE = False
    print("H2O not installed — simulated output shown.")
    print("Install: pip install h2o  (requires Java 8+)")

# Synthetic customer churn dataset
np.random.seed(42)
N = 2000

data = {
    'age':      np.random.randint(18, 75, N),
    'income':   np.round(np.random.exponential(50000, N), 0),
    'tenure':   np.random.randint(0, 120, N),
    'usage':    np.random.poisson(30, N),
    'calls':    np.random.poisson(5, N),
    'category': np.random.choice(['A', 'B', 'C', 'D'], N),
    'region':   np.random.choice(['North', 'South', 'East', 'West'], N),
}
churn_score = (
    -0.3 * (data['usage'] - 30) / 30
    + 0.2 * (data['income'] / 50000 - 1)
    + np.random.normal(0, 0.5, N)
)
data['churn'] = (churn_score > 0).astype(int)

if PANDAS_AVAILABLE:
    df_pd = pd.DataFrame(data)
    print(f"Dataset: {N:,} rows, {len(data)} columns")
    print(f"Churn rate: {df_pd['churn'].mean():.1%}")
    print(df_pd.head())

## Core Concept 1: H2O Cluster — The Engine

H2O runs as a **local or distributed cluster** (Java-based).  
`h2o.init()` starts a local H2O instance on your machine.  
On a Hadoop/Spark cluster, it starts workers on each node.

In [ ]:
if H2O_AVAILABLE:
    # Start a local H2O cluster
    h2o.init(
        nthreads=-1,        # use all CPU threads (-1 = all)
        max_mem_size='4g',  # allocate 4GB to H2O JVM
    )
    print(f"H2O cluster started: {h2o.cluster().cloud_name}")
    print(f"Nodes: {h2o.cluster().cloud_size}")

    # Convert pandas DataFrame to H2OFrame
    hf = h2o.H2OFrame(df_pd)

    # Tell H2O that 'category', 'region', 'churn' are categoricals
    hf['category'] = hf['category'].asfactor()
    hf['region']   = hf['region'].asfactor()
    hf['churn']    = hf['churn'].asfactor()  # CRITICAL: factor = classification

    print("\nH2OFrame info:")
    print(f"  Shape: {hf.shape}")
    print(f"  Types: {hf.types}")
    hf.head(5)

else:
    print("H2O cluster setup (simulated):")
    print()
    print("  import h2o")
    print("  h2o.init(nthreads=-1, max_mem_size='4g')")
    print()
    print("  H2O started — JVM running at http://localhost:54321")
    print("  Nodes: 1  (local mode)")
    print("  Threads: 8  (all cores)")
    print("  Memory: 4.0 GB")
    print()
    print("  # Convert pandas to H2OFrame")
    print("  hf = h2o.H2OFrame(df_pd)")
    print("  hf['churn'] = hf['churn'].asfactor()  # must mark as factor for classification")
    print("  hf['region'] = hf['region'].asfactor()  # categorical features")

## Core Concept 2: H2OAutoML — Automatic Model Training

H2OAutoML trains multiple algorithms and their stacked ensembles automatically.

In [ ]:
if H2O_AVAILABLE and PANDAS_AVAILABLE:
    # Split data
    train, test = hf.split_frame(ratios=[0.8], seed=42)
    print(f"Train: {train.shape}, Test: {test.shape}")

    x_cols = [c for c in hf.columns if c != 'churn']  # feature columns
    y_col  = 'churn'

    # Run AutoML
    t0 = time.time()
    aml = H2OAutoML(
        max_models=10,            # train at most 10 models
        max_runtime_secs=60,      # stop after 60 seconds
        seed=42,
        sort_metric='AUC',        # rank leaderboard by AUC
        verbosity='warn',
    )
    aml.train(
        x=x_cols,
        y=y_col,
        training_frame=train,
        leaderboard_frame=test,   # evaluate on held-out set
    )
    print(f"AutoML complete in {time.time()-t0:.0f}s")

    # Leaderboard
    print("\nLeaderboard (top 10 models):")
    lb = aml.leaderboard
    lb.head(10)

    # Best model
    leader = aml.leader
    print(f"\nBest model: {leader.model_id}")
    print(f"Best AUC:   {leader.auc(xval=True):.4f}")

else:
    print("H2OAutoML (simulated):")
    print()
    print("  aml = H2OAutoML(")
    print("      max_models=10,")
    print("      max_runtime_secs=60,")
    print("      seed=42,")
    print("      sort_metric='AUC',")
    print("  )")
    print("  aml.train(x=features, y='churn', training_frame=train)")
    print()
    print("  Leaderboard:")
    print("  model_id                                    auc    logloss")
    print("  StackedEnsemble_AllModels_1_AutoML          0.896   0.431")
    print("  StackedEnsemble_BestOfFamily_1_AutoML       0.893   0.434")
    print("  GBM_1_AutoML                                0.887   0.441")
    print("  XGBoost_1_AutoML                            0.884   0.444")
    print("  GBM_2_AutoML                                0.881   0.448")
    print("  DRF_1_AutoML                                0.872   0.459")
    print("  GLM_1_AutoML                                0.821   0.493")
    print("  DeepLearning_1_AutoML                       0.815   0.501")

## Core Concept 3: Predictions and Evaluation

In [ ]:
if H2O_AVAILABLE and PANDAS_AVAILABLE:
    # Predict with best model
    preds = leader.predict(test)
    print("Predictions (H2OFrame):")
    preds.head(5)

    # Model performance
    perf = leader.model_performance(test)
    print(f"\nModel performance on test set:")
    print(f"  AUC:      {perf.auc():.4f}")
    print(f"  Log loss: {perf.logloss():.4f}")
    print(f"  Accuracy: {perf.accuracy()[0][1]:.4f}")

    # Confusion matrix
    print(f"\nConfusion matrix:")
    print(perf.confusion_matrix())

    # Convert preds to pandas
    preds_pd = preds.as_data_frame()
    print(f"\nPredictions as pandas:\n{preds_pd.head()}")

else:
    print("Predictions and evaluation (simulated):")
    print()
    print("  preds = leader.predict(test)")
    print()
    print("  preds.head():")
    print("  predict   p0       p1")
    print("  0         0.712    0.288")
    print("  1         0.234    0.766")
    print("  0         0.678    0.322")
    print()
    print("  perf = leader.model_performance(test)")
    print("  AUC:       0.896")
    print("  Log loss:  0.431")
    print("  Accuracy:  0.834")
    print()
    print("  Confusion matrix:")
    print("          Predicted 0   Predicted 1")
    print("  Actual 0:    287           43")
    print("  Actual 1:     23          47")

## Core Concept 4: MOJO Export — Deploy Without H2O

**MOJO** (Model Object, Optimized) is H2O's model export format.  
A MOJO is a `.zip` file containing a fast Java model — you can serve it without H2O installed.  
This is H2O's key advantage for production deployment.

In [ ]:
import os, tempfile

if H2O_AVAILABLE and PANDAS_AVAILABLE:
    export_dir = tempfile.mkdtemp()

    # Export as MOJO
    mojo_path = leader.save_mojo(path=export_dir)
    print(f"MOJO saved to: {mojo_path}")
    print(f"Size: {os.path.getsize(mojo_path) / 1024:.1f} KB")
    print()

    # Export as POJO (plain Java object)
    pojo_path = leader.save_pojo(path=export_dir, get_jar=True)
    print(f"POJO saved to: {pojo_path}")

    # Download H2O scoring jar (needed to use MOJO)
    print("\nTo use MOJO in Java:")
    print("  java -cp h2o-genmodel.jar hex.genmodel.tools.PredictCsv \\")
    print("       --mojo model.zip --input test.csv --output predictions.csv")

    # Use MOJO in Python (import from h2o)
    mojo_model = h2o.import_mojo(mojo_path)
    mojo_preds = mojo_model.predict(test)
    print(f"\nMOJO predictions match original: checking...")

    import shutil
    shutil.rmtree(export_dir, ignore_errors=True)

else:
    print("MOJO export (simulated):")
    print()
    print("  # Export model as MOJO (no H2O needed at serving time)")
    print("  mojo_path = leader.save_mojo(path='/tmp/models/')")
    print("  # → /tmp/models/StackedEnsemble_AllModels_1_AutoML.zip")
    print()
    print("  # Serve with Java (no Python needed):")
    print("  java -cp h2o-genmodel.jar hex.genmodel.tools.PredictCsv \\")
    print("       --mojo model.zip --input new_customers.csv")
    print()
    print("  # Or use in Python:")
    print("  mojo_model = h2o.import_mojo('model.zip')")
    print("  mojo_model.predict(new_data)")
    print()
    print("  MOJO advantages:")
    print("    - No H2O dependency at serving time")
    print("    - Very fast Java inference")
    print("    - Works in microservices (add h2o-genmodel.jar to classpath)")
    print("    - Supports GBM, DRF, GLM, Deep Learning, XGBoost, stacked ensembles")

## Core Concept 5: Individual H2O Models (Manual Control)

When you need more control than AutoML, use H2O's individual model APIs.

In [ ]:
if H2O_AVAILABLE and PANDAS_AVAILABLE:
    from h2o.estimators.gbm import H2OGradientBoostingEstimator
    from h2o.estimators.random_forest import H2ORandomForestEstimator
    from h2o.grid.grid_search import H2OGridSearch

    # Train a GBM with specific hyperparameters
    gbm = H2OGradientBoostingEstimator(
        ntrees=100,
        max_depth=5,
        learn_rate=0.05,
        sample_rate=0.8,
        col_sample_rate=0.8,
        nfolds=5,              # cross-validation built in
        keep_cross_validation_predictions=True,
        seed=42,
    )
    gbm.train(x=x_cols, y=y_col, training_frame=train)

    print(f"GBM AUC (CV): {gbm.auc(xval=True):.4f}")
    print(f"GBM AUC (test): {gbm.model_performance(test).auc():.4f}")

    # Variable importance
    print("\nVariable importance:")
    vi = gbm.varimp(use_pandas=True)
    print(vi[['variable', 'relative_importance', 'percentage']].head())

    # Grid search
    print("\nGrid search over GBM hyperparameters:")
    hyper_params = {
        'max_depth': [3, 5, 7],
        'learn_rate': [0.01, 0.05, 0.1],
    }
    search_crit = {'strategy': 'RandomDiscrete', 'max_models': 6, 'seed': 42}

    grid = H2OGridSearch(
        model=H2OGradientBoostingEstimator(ntrees=50, seed=42),
        hyper_params=hyper_params,
        search_criteria=search_crit,
    )
    grid.train(x=x_cols, y=y_col, training_frame=train)

    grid_df = grid.get_grid(sort_by='auc', decreasing=True)
    print(grid_df)

else:
    print("Individual H2O models (simulated):")
    print()
    print("  from h2o.estimators.gbm import H2OGradientBoostingEstimator")
    print()
    print("  gbm = H2OGradientBoostingEstimator(")
    print("      ntrees=100, max_depth=5, learn_rate=0.05,")
    print("      nfolds=5, keep_cross_validation_predictions=True,")
    print("  )")
    print("  gbm.train(x=features, y='churn', training_frame=train)")
    print()
    print("  GBM AUC (CV):   0.881")
    print("  GBM AUC (test): 0.884")
    print()
    print("  Variable importance:")
    print("    usage:    0.354  (most important)")
    print("    income:   0.218")
    print("    tenure:   0.145")
    print("    age:      0.112")
    print()
    print("  Grid search → best: max_depth=5, learn_rate=0.05 → AUC=0.887")

## Core Concept 6: Model Explainability

H2O has built-in model explanation tools: SHAP, partial dependence plots, variable importance.

In [ ]:
if H2O_AVAILABLE and PANDAS_AVAILABLE:
    # SHAP contributions for individual predictions
    # (Available for tree models: GBM, DRF, XGBoost)
    contributions = leader.predict_contributions(test)
    contrib_pd = contributions.as_data_frame()
    print("SHAP contributions (first row):")
    print(contrib_pd.head(1).to_string())
    print()

    # Model explanation report (generates HTML)
    # h2o.explain(leader, test)  # opens interactive HTML
    print("H2O Explain:")
    print("  h2o.explain(leader, test)")
    print("  # Generates interactive HTML with:")
    print("  #   - SHAP summary plot")
    print("  #   - Partial dependence plots for each feature")
    print("  #   - Model leaderboard comparison")
    print("  #   - Variable importance")

    # Shut down H2O
    h2o.shutdown(prompt=False)

else:
    print("Model explainability (simulated):")
    print()
    print("  # SHAP contributions (row-level explanations)")
    print("  contribs = leader.predict_contributions(test)")
    print("  # Returns a H2OFrame with one SHAP value per feature per row")
    print()
    print("  Row 0 SHAP contributions:")
    print("    usage:    +0.234  (higher usage → lower churn)")
    print("    income:   -0.187  (higher income → higher churn here)")
    print("    tenure:   +0.098  (longer tenure → lower churn)")
    print("    BiasTerm: +0.012")
    print()
    print("  # Full explainability report (opens browser)")
    print("  h2o.explain(leader, test)")
    print("  # → Interactive HTML with SHAP, PDP, variable importance")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Forgetting `.asfactor()` | Regression instead of classification | Always mark target and categoricals with `.asfactor()` |
| Java not installed | `RuntimeError: H2O failed to start` | Install Java 8+ and set `JAVA_HOME` |
| `h2o.init()` called twice | Warning: cluster already running | Use `h2o.connect()` to reuse existing cluster |
| Not shutting down H2O | Memory leak in long sessions | Always call `h2o.shutdown(prompt=False)` when done |
| Running AutoML too long | OOM / JVM crash | Set `max_runtime_secs` and `max_models` |
| Collecting entire H2OFrame | OOM in pandas | Use `.as_data_frame()` only on small subsets |

## Mini Project: H2O AutoML vs Manual GBM

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

if PANDAS_AVAILABLE:
    print("=" * 60)
    print("H2O AutoML vs MANUAL GBM COMPARISON")
    print("=" * 60)
    print()

    # Prepare data for sklearn
    df_sk = df_pd.copy()
    for col in ['category', 'region']:
        df_sk[col] = LabelEncoder().fit_transform(df_sk[col])

    X = df_sk.drop('churn', axis=1)
    y = df_sk['churn']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    # Manual GBM (default settings)
    t0 = time.time()
    gbm_manual = GradientBoostingClassifier(random_state=42)
    gbm_manual.fit(X_tr, y_tr)
    auc_manual = roc_auc_score(y_te, gbm_manual.predict_proba(X_te)[:, 1])
    print(f"  Manual GBM (default):     AUC = {auc_manual:.4f}  ({time.time()-t0:.1f}s)")

    # Manual GBM (tuned)
    t0 = time.time()
    gbm_tuned = GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
    gbm_tuned.fit(X_tr, y_tr)
    auc_tuned = roc_auc_score(y_te, gbm_tuned.predict_proba(X_te)[:, 1])
    print(f"  Manual GBM (tuned):       AUC = {auc_tuned:.4f}  ({time.time()-t0:.1f}s)")

    print()
    if H2O_AVAILABLE:
        print("  H2O AutoML result: see leaderboard above")
    else:
        print("  H2O AutoML (simulated):   AUC = 0.8960  (StackedEnsemble wins!)")

    print()
    print("  Key insight: H2O AutoML's stacked ensemble beats any single model")
    print("  because it combines the strengths of multiple diverse algorithms.")

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "How does H2O AutoML work internally?",
     "a": """H2O AutoML runs a pre-defined sequence of algorithms:

Phase 1 — Default models (fast, diverse):
  XGBoost (3 default configs)
  GBM (5 configs: default, low learning rate, high depth, ...)
  DRF (Distributed Random Forest)
  GLM (L2 regularization)
  Deep Learning (1 default)

Phase 2 — Tuning (if time_limit allows):
  Random grid search over hyperparameters for GBM and XGBoost

Phase 3 — Stacking:
  StackedEnsemble_BestOfFamily: best model from each algorithm family
  StackedEnsemble_AllModels: all base models
  (StackedEnsembles almost always top the leaderboard)

Cross-validation:
  All models trained with 5-fold CV by default.
  OOF predictions used for stacking (no leakage).

Leaderboard sorted by: AUC (classification), RMSE (regression)."""},

    {"q": "What is MOJO and why is it important for production?",
     "a": """MOJO (Model Object, Optimized) is H2O's deployable model format.

What it is:
  A .zip file containing the model in optimized Java bytecode.
  Supports: GBM, DRF, GLM, Deep Learning, XGBoost, Stacked Ensembles.

Why important:
  1. No H2O dependency at serving time:
     You only need h2o-genmodel.jar (tiny, ~100MB).
     Your production Java service doesn't need H2O running.

  2. Very fast inference:
     MOJO compiles to optimized JVM bytecode.
     Single predictions in microseconds.

  3. Portable:
     Works in any Java environment (Spark, Kafka, microservices).
     Can be embedded in Android, edge devices.

  4. Reproducible:
     The .zip contains all preprocessing: encoding, normalization, splits.
     No 'training-serving skew' — same transformations guaranteed.

Export:
  leader.save_mojo('model.zip')

Serve in Java:
  EasyPredictModelWrapper model = new EasyPredictModelWrapper(MojoModel.load('model.zip'));
  BinomialModelPrediction pred = model.predictBinomial(row);"""},

    {"q": "H2O AutoML vs AutoGluon — which would you choose?",
     "a": """Both are excellent. The choice depends on requirements:

Choose H2O AutoML when:
  - Production serving in Java/Scala (MOJO export)
  - Existing Spark/Hadoop infrastructure (H2O Sparkling Water)
  - Need Explainability (built-in SHAP, PDP, ICE)
  - Enterprise support is needed (H2O.ai paid plans)
  - R is your primary language (H2O has excellent R API)

Choose AutoGluon when:
  - Maximum accuracy is the primary goal
  - Multimodal data (text + tabular + images)
  - Python-only environment
  - Research / Kaggle competitions
  - Stack ensembling depth matters (AutoGluon goes deeper)

Accuracy comparison:
  Both win Kaggle competitions. AutoGluon tends to perform slightly
  better on tabular benchmarks (deeper stacking).
  H2O wins on deployment ease (MOJO) and interpretability.

Rule:
  Production Java serving → H2O
  Best Python accuracy → AutoGluon
  Enterprise + explainability → H2O (Driverless AI)"""},

    {"q": "How does H2O stacked ensembling work?",
     "a": """H2O stacking follows the same principle as AutoGluon:

1. Base models trained with 5-fold cross-validation.
2. Out-of-fold (OOF) predictions collected — each prediction made
   on data the model never saw during training.
3. Meta-learner (GLM by default) trained on OOF predictions.
4. At test time: base models predict → meta-learner combines.

H2O has two stacked ensembles:
  StackedEnsemble_BestOfFamily:
    Takes the best model from each algorithm family
    (best GBM + best XGBoost + best DRF + GLM + NN)
    → More diverse, often better generalization

  StackedEnsemble_AllModels:
    All trained models as base learners
    → Higher risk of overfitting if many models
    → Sometimes better if all models are good

Meta-learner:
  Default: GLM with non-negative weights (ensures non-negative combination)
  Can change to: GBM, DRF for the meta-learner itself

Why it works:
  Each model has different biases — GBM underestimates tails,
  RF overestimates boundaries. Stacking compensates each other."""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Step | H2O API |
|------|---------|
| Start cluster | `h2o.init(nthreads=-1, max_mem_size='4g')` |
| Load data | `h2o.H2OFrame(pandas_df)` or `h2o.import_file('data.csv')` |
| Mark categorical | `hf['col'] = hf['col'].asfactor()` |
| Train/test split | `train, test = hf.split_frame(ratios=[0.8], seed=42)` |
| Run AutoML | `H2OAutoML(max_models=10, max_runtime_secs=60).train(x, y, train)` |
| View leaderboard | `aml.leaderboard` |
| Best model | `aml.leader` |
| Predict | `leader.predict(test)` |
| Evaluate | `leader.model_performance(test).auc()` |
| Export MOJO | `leader.save_mojo('model.zip')` |
| SHAP values | `leader.predict_contributions(test)` |
| Explain | `h2o.explain(leader, test)` |
| Shutdown | `h2o.shutdown(prompt=False)` |

### Next Steps
1. **H2O AutoML docs**: [https://docs.h2o.ai/h2o/latest-stable/h2o-docs/automl.html](https://docs.h2o.ai/h2o/latest-stable/h2o-docs/automl.html)
2. **H2O Tutorials**: [https://github.com/h2oai/h2o-tutorials](https://github.com/h2oai/h2o-tutorials)
3. **Next**: Optuna — flexible hyperparameter optimization for any ML framework